# 01 — Data Cleaning

**Purpose:** upstream data pipeline — turns the raw GMT export into the single
analysis dataset (`results_clean_runs.csv`). Feeds the thesis *Methodology /
Data cleaning* section.

Pipeline:
1. Load raw `results_linux.csv`
2. Detect & remove outliers (**boxplot / 1.5×IQR fence**, per language × benchmark group, on CPU energy + execution time)
3. Convert units to human-readable values
4. Export `results_clean_runs.csv`

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))   # make the shared style module importable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import importlib
import plot_style as ps
importlib.reload(ps)   # pick up edits to plot_style.py without a kernel restart
ps.apply_style()

RAW_CSV  = Path('../results/results_linux.csv')
RUNS_CSV = Path('../results/results_clean_runs.csv')

## 1. Load raw data

In [ ]:
df = pd.read_csv(RAW_CSV)
print(f'Shape: {df.shape}')
df.head(3)

## 2. Outlier detection & removal (Boxplot / 1.5×IQR fence)

Outliers are identified using the standard boxplot rule: any run whose value falls **outside [Q1 − 1.5×IQR, Q3 + 1.5×IQR]** is flagged.  
Detection is applied **per (language × benchmark) group** on the two primary metrics — CPU energy and execution time — to avoid over-flagging from noisy ancillary columns (disk, network, etc.).

In [ ]:
# Numeric metric columns only (exclude identifiers)
ID_COLS     = ['run_id', 'measured_at', 'language', 'benchmark']
METRIC_COLS = [c for c in df.columns if c not in ID_COLS]

# Detect outliers only on the two primary benchmarking signals
KEY_METRICS = [c for c in METRIC_COLS if 'cpu_energy' in c or 'phase_time' in c]
print('Outlier detection on:', KEY_METRICS)

def boxplot_outlier_mask(group: pd.DataFrame, cols: list[str]) -> pd.Series:
    """
    Standard boxplot fence: flag rows outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
    on any of the given columns.
    """
    outlier = pd.Series(False, index=group.index)
    for col in cols:
        q1, q3 = group[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:          # constant column in this group — skip
            continue
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier |= (group[col] < lo) | (group[col] > hi)
    return outlier

outlier_flags = (
    df.groupby(['language', 'benchmark'], group_keys=False)
      .apply(lambda g: boxplot_outlier_mask(g, KEY_METRICS))
)

n_outliers = outlier_flags.sum()
print(f'Outlier runs flagged : {n_outliers} / {len(df)}  ({n_outliers/len(df)*100:.1f}%)')

df_clean = df[~outlier_flags].copy()
print(f'Rows after removal   : {len(df_clean)}')

In [ ]:
# Visualise the 1.5×IQR DETECTION on the real (raw) data — two separate figures,
# one for CPU energy and one for execution time. Each figure is its own thing:
# the box/whiskers are computed on the FULL raw distribution (so the whiskers
# sit exactly at the 1.5×IQR fence), normal runs are scatter-dotted in a single
# neutral colour, and the runs flagged on THAT metric are drawn as red ✗ beyond
# the whiskers. The two figures use independent per-metric flags — they do NOT
# share values, and compiler is not encoded here (this is a detection view).
from matplotlib.lines import Line2D

COL_E      = [c for c in METRIC_COLS if 'cpu_energy' in c][0]   # µJ
COL_T      = [c for c in METRIC_COLS if 'phase_time' in c][0]   # µs
benchmarks = sorted(df['benchmark'].unique())

# Neutral (non-compiler) styling for the detection figures
BOX_FILL = '#b0c4de'   # light steel blue box tint
DOT_CLR  = '#4c72b0'   # calm blue for the individual run dots

def metric_flags(col):
    """
    Per-(language × benchmark) 1.5×IQR outlier flag for a single metric column.
    Reuses boxplot_outlier_mask from the detection cell. Returns a boolean Series
    aligned to df.index (True = outlier on `col`).
    """
    return df.groupby(['language', 'benchmark'], group_keys=False) \
             .apply(lambda g: boxplot_outlier_mask(g, [col]))

def _draw_metric_panel(ax, sub, langs, col, divisor, ylabel):
    """
    Draw one boxplot DETECTION panel for a single (benchmark × metric).
    Inputs: ax (target axes); sub (rows for this benchmark, carrying a boolean
    '_outlier' column — the flag for THIS figure's metric); langs (x order);
    col + divisor (metric column and its raw→display unit divisor); ylabel.
    The box/whiskers are computed on the FULL raw data (whiskers = 1.5×IQR
    fence); normal runs are scatter-dotted in a single neutral colour and
    detected outliers are drawn as red ✗ beyond the whiskers. Returns the count.
    """
    data_all     = [sub[sub['language'] == l][col].values / divisor for l in langs]            # box on raw data
    data_normal  = [sub[(sub['language'] == l) & ~sub['_outlier']][col].values / divisor for l in langs]
    data_outlier = [sub[(sub['language'] == l) &  sub['_outlier']][col].values / divisor for l in langs]

    bp = ax.boxplot(
        data_all, labels=langs, patch_artist=True, showfliers=False, whis=1.5,
        medianprops=dict(color='black', linewidth=1.5),
        whiskerprops=dict(color='#555555'),
        capprops=dict(color='#555555'),
    )
    for patch in bp['boxes']:
        patch.set_facecolor(BOX_FILL)
        patch.set_alpha(0.5)

    for j, lang in enumerate(langs):
        x = j + 1
        ax.scatter([x] * len(data_normal[j]),  data_normal[j],
                   color=DOT_CLR, s=10, alpha=0.6, zorder=3)
        ax.scatter([x] * len(data_outlier[j]), data_outlier[j],
                   color='red', s=24, alpha=0.9, marker='x', zorder=4, linewidths=1.3)

    ax.set_yscale('log')
    ax.set_ylabel(ylabel, fontsize=7)
    ax.tick_params(axis='x', rotation=45, labelsize=6)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    return sum(len(v) for v in data_outlier)

def outlier_figure(col, divisor, ylabel, metric_label, flags, save_name):
    """
    Build and save one per-benchmark 1.5×IQR detection figure for a single
    metric. Inputs: col + divisor (metric column and raw→display unit divisor);
    ylabel; metric_label (panel/suptitle wording); flags (boolean Series, the
    per-metric outlier mask for THIS figure); save_name (figure stem for
    ps.save_fig). Boxes drawn on raw data; this metric's outliers in red. The
    legend shows only the outlier marker. Returns the matplotlib Figure.
    """
    ncols = 4
    nrows = int(np.ceil(len(benchmarks) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
    axes = axes.flatten()

    for i, bm in enumerate(benchmarks):
        sub = df[df['benchmark'] == bm].copy()
        sub['_outlier'] = flags.reindex(sub.index, fill_value=False)
        langs = sorted(sub['language'].unique())
        n_out = _draw_metric_panel(axes[i], sub, langs, col, divisor, ylabel)
        axes[i].set_title(f'{bm} · {n_out} outliers', fontsize=9)

    for k in range(len(benchmarks), len(axes)):
        axes[k].set_visible(False)

    # Legend: outlier marker only
    legend_handles = [Line2D([0], [0], marker='x', color='red', markersize=8, markeredgewidth=1.5,
                             label=f'Outlier (1.5×IQR fence on {metric_label.lower()})', linestyle='None')]
    fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, -0.02),
               ncol=1, fontsize=10, frameon=True)

    plt.tight_layout()
    ps.save_fig(fig, save_name)
    return fig

energy_flags = metric_flags(COL_E)
time_flags   = metric_flags(COL_T)

# Figure 1 — CPU energy detection (µJ -> J)
outlier_figure(COL_E, 1e6, 'CPU Energy (J, log)', 'CPU Energy', energy_flags, '01_outlier_boxplots_cpu_energy')
plt.show()

# Figure 2 — execution time detection (µs -> s)
outlier_figure(COL_T, 1e6, 'Time (s, log)', 'Execution Time', time_flags, '01_outlier_boxplots_time')
plt.show()

> **Takeaway:** the 1.5×IQR fence flags 160/1440 runs (11.1%) as outliers, mostly
> isolated spikes rather than whole groups — confirming the cleaning is conservative
> and per-cell, not discarding any (language × benchmark) combination wholesale.

## 2b. Cleaning accuracy — within-group dispersion

Proof that the cleaning tightened the data. We use the **coefficient of variation**
(CV) — the standard deviation relative to the mean — as a *unit-free* measure of how
much the repeated runs of a given (language × benchmark) cell disagree. Lower CV =
higher measurement reproducibility.

**How the values are computed**

1. **Per cell.** For each of the 144 (language × benchmark) cells, across its ~10
   repeated runs, compute
   $$\mathrm{CV} = \frac{\sigma}{\mu}\times 100\%$$
   where $\mu$ is the mean and $\sigma$ the sample standard deviation (`std(ddof=1)`)
   of that metric over the cell's runs.
2. **Average.** Take the mean of those 144 cell-CVs (every cell weighted equally) →
   the **Mean CV** reported in the table.
3. **Before vs after.** Steps 1–2 are run twice: on the raw data `df` (with outliers)
   → *Mean CV before*; and on the cleaned data `df_clean` (after the 1.5×IQR removal)
   → *Mean CV after*.
4. **Reduction.** The relative drop in dispersion:
   $$\text{Reduction} = \frac{\mathrm{CV}_{\text{before}} - \mathrm{CV}_{\text{after}}}{\mathrm{CV}_{\text{before}}}\times 100\%$$

Because CV is unit-free, the three metrics are directly comparable despite their
different units (J, s). A single mean per metric is reported (rather than one value
per cell) because each cell's CV rests on only ~10 runs and is noisy; the mean over
144 cells is the stable, slide-ready summary.

*Note: CPU energy and execution time are the outlier-detection criteria; memory
energy is a supplementary check (not used for detection), which is why it improves
the least.*

In [ ]:
from pathlib import Path
OUTPUTS_DIR = Path('outputs'); OUTPUTS_DIR.mkdir(exist_ok=True)

# Coefficient of variation (std/mean) per (language × benchmark) cell, averaged
# over all cells — a unit-free measure of within-group dispersion. Lower = the
# repeated runs agree more closely (higher measurement reproducibility).
CV_METRICS = [
    ('CPU Energy',     'cpu_energy_rapl_msr_component-package_0-uj'),
    ('Execution Time', 'phase_time_syscall_system-system-us'),
    ('Memory Energy',  'memory_energy_rapl_msr_component-dram_0-uj'),
]

def mean_cv(data, col):
    """Mean within-(language × benchmark) coefficient of variation (%) for `col`."""
    g = data.groupby(['language', 'benchmark'])[col]
    return (g.std(ddof=1) / g.mean()).mean() * 100

rows = []
for name, col in CV_METRICS:
    before = mean_cv(df, col)          # raw, with outliers
    after  = mean_cv(df_clean, col)    # after 1.5×IQR removal
    rows.append({
        'Metric': name,
        'Mean CV before (%)': round(before, 2),
        'Mean CV after (%)':  round(after, 2),
        'Reduction (%)':      round((before - after) / before * 100, 1),
    })
cv_table = pd.DataFrame(rows).set_index('Metric')
cv_table.to_csv(OUTPUTS_DIR / 'cleaning_accuracy_cv.csv')
print('Saved → outputs/cleaning_accuracy_cv.csv')

# Display-formatted copy (strings) for the styled table figure (PNG + PDF).
disp = pd.DataFrame({
    'Mean CV before': cv_table['Mean CV before (%)'].map(lambda v: f'{v:.2f}%'),
    'Mean CV after':  cv_table['Mean CV after (%)'].map(lambda v: f'{v:.2f}%'),
    'Reduction':      cv_table['Reduction (%)'].map(lambda v: f'{v:.1f}%'),
}, index=cv_table.index)
ps.styled_table_fig(
    disp,
    'Measurement Reproducibility — mean within-group CV before vs after cleaning',
    '01_cleaning_accuracy_cv',
    highlight_col='Reduction',
)
cv_table

In [ ]:
# Per-group count after cleaning — flag any group with fewer than 5 clean runs
counts = df_clean.groupby(['language', 'benchmark']).size().rename('n_runs')
low = counts[counts < 5]
if len(low):
    print('⚠ Groups with < 5 clean runs:')
    print(low.to_string())
else:
    print('All groups have ≥ 5 clean runs.')
counts.unstack('benchmark').fillna(0).astype(int)

## 3. Unit conversions

In [ ]:
# Conversion rules keyed by column-name suffix.
# Each entry: (divisor, new_unit_label)
CONVERSIONS = {
    '-ug' : (1e6,    'g'),        # micro-grams  → grams
    '-uj' : (1e6,    'J'),        # micro-Joules → Joules
    '-mw' : (1e3,    'W'),        # milli-Watts  → Watts
    '-us' : (1e6,    's'),        # micro-seconds → seconds
    '-bytes': (1e6,  'MB'),       # bytes → MB  (disk/net totals)
}

rename_map = {}
for col in METRIC_COLS:
    for suffix, (divisor, unit) in CONVERSIONS.items():
        if col.endswith(suffix):
            base = col[: -len(suffix)]
            new_col = f'{base}-{unit.lower()}'
            df_clean[new_col] = df_clean[col] / divisor
            rename_map[col] = new_col
            break

# Drop the original raw-unit columns
df_clean = df_clean.drop(columns=list(rename_map.keys()))

METRIC_COLS_CONV = [c for c in df_clean.columns if c not in ID_COLS]
print('Converted columns:')
for old, new in rename_map.items():
    print(f'  {old}  →  {new}')

## 4. Export

In [ ]:
df_clean.to_csv(RUNS_CSV, index=False)
print(f'Saved → {RUNS_CSV}')